In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from glob import glob
# Aplicar configuraciones de visualización total de Pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)


import warnings
warnings.filterwarnings("ignore")

import func.func_gcp as bq

In [3]:
ruta_dim_local = r"bk\input\dim\dim_locales.xlsx"
ruta_dim_secciones = r"bk\input\dim\dim_secciones.xlsx"
ruta_dim_puestos = r"bk\input\dim\Mapeo_Puestos_Dotacion_Tipos.xlsx"

In [4]:
df_local = pd.read_excel(ruta_dim_local)
df_secc = pd.read_excel(ruta_dim_secciones)
df_puestos = pd.read_excel(ruta_dim_puestos)

In [4]:
df_local1 = df_local[["codigo_local", "nombre_Kr", "nombre_Dot", "nombre_Dot2-", "nombre_Fin"]]
df_local1

,codigo_local,nombre_Kr,nombre_Dot,nombre_Dot2-,nombre_Fin
0,P001,CFR PVEA LOS OLIVOS,FRET - Pvea Los Olivos,Fret Pvea Los Olivos,Los Olivos - PVH
1,SO02,SPO PVO PUCALLPA,Spo - Pvo Pucallpa,Pvo - Pucallpa,Pucallpa - PVO
2,SO03,SPO PVO HUANUCO,Spo - Pvo Huanuco,Pvo - Huanuco,Huanuco - PVO
3,SO04,SPO PVO JAEN,Spo - Pvo Jaen,Pvo - Jaen,Jaen - PVO
4,P010,CFR SVEA CAMINOS DEL INCA,FRET - Svea Caminos Del Inca,Fret Svea Caminos Del Inca,Caminos del Inca - PVS
...,...,...,...,...,...
114,P724,CFR PVEA TUMBES,FRET - Pvea Tumbes,Fret Pvea Tumbes,Tumbes - PVH
115,Z991,Back Office Plaza Vea,Back Office Plaza Vea,Back Office Plaza Vea,Back Office Plaza Vea
116,Z992,Back Office Vivanda,Back Office Vivanda,Back Office Vivanda,Back Office Vivanda
117,Z993,Retail Media - Pv,Retail Media - Pv,Retail Media - Pv,Retail Media - Pv


In [8]:
df_local1["codigo_local"].duplicated().any()

np.False_

In [9]:
duplicados = df_local1.loc[
    df_local1["codigo_local"].duplicated(keep=False)
].sort_values("codigo_local")

duplicados

,codigo_local,nombre_Kr,nombre_Dot,nombre_Dot2-,nombre_Fin


In [5]:
df_secc

,TIPO,_Es_Total,_Area,_ConsideraSeccion
0,CAJAS,Individual,Caja,Si
1,FRESCOS,Individual,Fresc,Si
2,ABARROTES,Individual,Abarr,Si
3,RECEPCIÓN,Individual,Recep,Si
4,INVENTARIOS,Individual,Invent,Si
5,ALMACÉN FRESCOS,Individual,Alm Fr,Si
6,ALMACÉN,Individual,Alm Ab,Si
7,ELECTRO,Individual,Electro,Si
8,SIN ELECTRO,Grupo,S/Elec,Si
9,MULTIFUNCIONAL,Individual,Multi,Si


In [11]:
df_puestos.columns

Index(['Departamento', 'Nombre de Posición', 'Grupo_Area', 'Grupo_Puestos',
       'Subgrupo_Puesto', 'Campaña/Extra', 'Grupo_Velocidad', 'Puesto_Reporte',
       'BonoMensual', 'BonoMensual_Tipo', 'BonoRT', 'BonoPoli', 'BonoMedio',
       'BonoAlto', 'Modalidad', 'Fuente', 'Cuenta',
       'ConsideradoEnMaestroAnterior', 'Agregado_en', 'Dep+Pos',
       'BuscaMaestroBI', 'BuscaMaestroBI-Total'],
      dtype='object')

In [16]:
import re
import unicodedata

def clean_columns(df):
    df = df.copy()

    df.columns = [
        re.sub(
            r"[^a-z0-9]+",
            "_",
            unicodedata.normalize("NFKD", col)
            .encode("ascii", "ignore")
            .decode("utf-8")
            .lower()
            .strip()
        ).strip("_")
        for col in df.columns
    ]

    return df

In [17]:
df_puestos1 = clean_columns(df_puestos)

In [6]:
bq.write_dataframe( df= df_secc, 
                   project_id="d-sfh-un-pvea", 
                   dataset_id="raw_operaciones", 
                   table_id="dim_secciones",
                   mode = "append"
                   )

Se cargaron 10 filas en d-sfh-un-pvea.raw_operaciones.dim_secciones (append)


In [7]:
bq.write_dataframe( df= df_local1, 
                   project_id="d-sfh-un-pvea", 
                   dataset_id="raw_operaciones", 
                   table_id="dim_dict_local",
                   mode = "append"
                   )

Se cargaron 119 filas en d-sfh-un-pvea.raw_operaciones.dim_dict_local (append)


In [18]:
bq.write_dataframe( df= df_puestos1, 
                   project_id="d-sfh-un-pvea", 
                   dataset_id="raw_operaciones", 
                   table_id="dim_puestos",
                   mode = "append"
                   )

Se cargaron 865 filas en d-sfh-un-pvea.raw_operaciones.dim_puestos (append)
